# Baseline BLEU-4 Sanity Check -- TRCaptionNet++ Large (dokunulmamis)

Bu notebook'un tek amaci: **kendi eval pipeline'imizin (train.py/trainer.py/eval.py + tools/*.py + metrik hesaplama)
dogru calisip calismadigini** bilinen, guvenilir bir modelle dogrulamak.

Burada egitim (fine-tuning) YAPILMIYOR. Model hic degistirilmiyor:

- **Encoder:** `DINOv2 ViT-L/14` (bizim degistirdigimiz MobileCLIP degil)
- **Decoder:** `BertLMHeadModel`, checkpoint'in kendi orijinal agirliklariyla
- **Checkpoint:** public `TRCaptionNetpp_Large.pth` (yayinlanmis, dokunulmamis)
- **Config:** `configs/tasviret/tasviretpp_large_tasviret.yaml` -- checkpoint'in kendi orijinal
  config'i (ayni alanlar `TRCaptionNetpp/app.py`'deki resmi demo ile birebir ayni: `dino2: dinov2_vitl14`,
  `bert: dbmdz/electra-base-turkish-mc4-cased-discriminator`)

`eval.py`, `trainer.py` icindeki gercek `eval()` metoduyla ayni `predict()` + `evaluate_on_coco_caption()`
fonksiyonlarini kullanir; herhangi bir `optimizer.step()` cagrilmadan, checkpoint'i oldugu gibi yukleyip
TasvirEt **test** split'inde caption uretir ve resmi COCO-caption metriklerini (Bleu_1..4, ROUGE_L, CIDEr) hesaplar.

Cikan Bleu_4'u, TRCaptionNet++ makalesindeki/resmi demodaki bilinen degerle karsilastirarak, pipeline'imizin
dogru olup olmadigina karar verecegiz.

## 1. Runtime kontrolu

Colab runtime'da GPU olarak (mumkunse) **L4** sec.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Repoyu klonla

`baseline-bleu4-sanity` branch'i, bu sanity-check icin gerekli olan minimal dosya kumesini icerir
(hibrit MobileCLIP deneylerinden bagimsiz, ayri/basit bir dal). Bu branch senin kendi
`github.com/mhr871/TRCAP` reponda -- once bu branch'i kendi makinenden `git push origin baseline-bleu4-sanity`
ile push etmis olman gerekir, aksi halde asagidaki clone bulamaz.

In [ ]:
%cd /content
!rm -rf /content/TRCAP
!git clone -b baseline-bleu4-sanity https://github.com/mhr871/TRCAP.git
%cd /content/TRCAP
!git rev-parse --short HEAD

## 3. Kutuphaneleri kur ve kontrol et

In [ ]:
!python -m pip install -r requirements_colab.txt

In [ ]:
!python -c "import torch, transformers, tokenizers, cv2, pyarrow; print('torch=', torch.__version__); print('transformers=', transformers.__version__); print('tokenizers=', tokenizers.__version__); print('opencv=', cv2.__version__); print('pyarrow=', pyarrow.__version__); print('cuda=', torch.cuda.is_available())"

## 4. Public checkpoint'i indir

Modele hic dokunmuyoruz -- sadece yayinlanmis agirliklari indiriyoruz.

In [ ]:
!python tools/download_checkpoint.py --output checkpoints/TRCaptionNetpp_Large.pth

In [ ]:
!sha256sum checkpoints/TRCaptionNetpp_Large.pth

Beklenen deger:

```text
c055ef247f968c86140b941506026721ca4c301ef3c7f6b421caec89ada8ebf3
```

## 5. Caption JSON ve split'leri hazirla

In [ ]:
!python tools/prepare_tasviret.py --allow-missing-images

Beklenen sayilar:

```text
train: 6000 images, 12028 captions
val:   1000 images,  2006 captions
test:  1000 images,  2003 captions
```

## 6. Eslesen Flickr8K goruntulerini indir

`atasoglu/flickr8k-turkish` mirror'inin sabitlenmis revizyonundan ~1.1 GB indirir.

In [ ]:
!python tools/download_tasviret_images.py

In [ ]:
!python tools/prepare_tasviret.py --images-root Data/flickr8k/images

## 7. Preflight kontrolu

GPU/VRAM, config, checkpoint SHA256, split sayilari, 8.000 goruntunun acilabilirligi ve modelin
`strict=True` yuklenmesini dogrular.

In [ ]:
!PYTHONPATH=/content/TRCAP python tools/preflight_colab.py

## 8. SANITY CHECK: public checkpoint'in kendi haliyle test BLEU-4'u

**Bu hucre bu notebook'un asil amaci.** Hicbir egitim/fine-tuning yapilmiyor; `eval.py` sadece
checkpoint'i `strict=True` ile oldugu gibi yukleyip TasvirEt **test** split'inde caption uretiyor ve
resmi metrikleri hesapliyor -- ayni `predict()`/`evaluate_on_coco_caption()` kodu bizim butun
deneylerimizde kullandigimiz kod.

In [ ]:
!PYTHONPATH=/content/TRCAP python eval.py   --config configs/tasviret/tasviretpp_large_tasviret.yaml   --weights checkpoints/TRCaptionNetpp_Large.pth   --test-json Data/tasvir-et/tasvir_test.json   --test-data Data/flickr8k/images   --dataset tasviret   --output-dir eval_outputs/tasviret_test_public_checkpoint

## 9. Sonuclari oku

In [ ]:
import json
with open('eval_outputs/tasviret_test_public_checkpoint/metrics.json', encoding='utf-8') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

In [ ]:
import json
with open('eval_outputs/tasviret_test_public_checkpoint/predictions.json', encoding='utf-8') as f:
    preds = json.load(f)
for p in preds[:10]:
    print(p)

## Yorum

- Buradaki `Bleu_4` degeri, bizim eval pipeline'imizin dogrulugunun referans noktasi.
- Eger uretilen caption'lar anlamli Turkce cumleler ise ve `Bleu_4` makul bir degerdeyse
  (rastgele/bozuk kelime yigini degil), pipeline'imiz (tokenization, generation, metrik hesaplama)
  guvenilir demektir -- o zaman kendi hibrit (MobileCLIP) deneylerimizdeki dusuk skorlarin nedeni
  pipeline degil, o deneylere ozgu bir sorundur (bkz. `devam.md`, `hibrit_0` altinda).
- Eger burada da caption'lar bozuksa/Bleu_4 anlamsizsa, sorun pipeline'in kendisinde (veri, tokenizer,
  metrik hesaplama) demektir ve butun deneylerimizi etkiliyor olabilir.